# Generating Creative Text (Stories, Poems) Using Language Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Generate creative stories
- Generate poems
- Use language models creatively
- Control generation style
- Evaluate creative outputs

## 🔗 Prerequisites

- ✅ Understanding of text generation
- ✅ Understanding of creative writing
- ✅ Language model knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 2**:
- Generating creative text (stories, poems) using language models
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 2 Practical Content

---

## Introduction

**Creative text generation** uses language models to produce stories, poems, and other creative content, demonstrating the artistic capabilities of generative AI.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import torch, torch.nn as nn, torch.optim as optim, numpy as np
print(f'PyTorch {torch.__version__}')
print('✅ Libraries imported!')
print('\nGenerating Creative Text: Stories and Poems')
print('=' * 60)
print('\nCreative Generation techniques demonstrated:')
print('  - Temperature sampling  (higher T → more random/creative)')
print('  - Top-k sampling        (only sample from top-k tokens)')
print('  - Character LSTM language model trained on a story corpus')

torch.manual_seed(7); np.random.seed(7)
corpus = (
    'once upon a time there was a brave knight who lived in a tall tower. '
    'every morning the knight would gaze at the distant mountains. '
    'the wind whispered secrets through the ancient trees. '
    'a dragon flew over the silver lake and breathed fire. '
    'roses are red violets are blue the moon shines bright on me and you. '
) * 10
chars = sorted(set(corpus)); c2i = {c: i for i, c in enumerate(chars)}
i2c  = {i: c for c, i in c2i.items()}; V, S = len(chars), 40

X = torch.tensor([[c2i[corpus[j+k]] for k in range(S)] for j in range(len(corpus)-S)], dtype=torch.long)
y = torch.tensor([c2i[corpus[j+S]] for j in range(len(corpus)-S)], dtype=torch.long)

class CharLM(nn.Module):
    def __init__(self): super().__init__(); self.e=nn.Embedding(V,64); self.l=nn.LSTM(64,256,batch_first=True); self.f=nn.Linear(256,V)
    def forward(self, x): return self.f(self.l(self.e(x))[0][:,-1,:])

model = CharLM(); opt = optim.Adam(model.parameters(), lr=3e-3); crit = nn.CrossEntropyLoss()
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(X, y), batch_size=128, shuffle=True)
for ep in range(20):
    model.train(); el=0
    for xb,yb in loader:
        opt.zero_grad(); loss=crit(model(xb),yb); loss.backward(); opt.step(); el+=loss.item()
    if (ep+1)%5==0: print(f'Epoch {ep+1}: loss={el/len(loader):.4f}')

def generate(seed, n=80, temperature=1.0, top_k=0):
    model.eval(); out=seed; ctx=[c2i.get(c,0) for c in seed[-S:]]
    for _ in range(n):
        x=torch.tensor([ctx],dtype=torch.long)
        with torch.no_grad(): logits=model(x)[0]/max(temperature,1e-6)
        if top_k>0:
            vals,idx=torch.topk(logits,top_k); mask=torch.full_like(logits,-1e9); mask[idx]=vals; logits=mask
        probs=torch.softmax(logits,0).numpy(); nxt=int(np.random.choice(V,p=probs))
        out+=i2c[nxt]; ctx=ctx[1:]+[nxt]
    return out

print('\n--- Temperature 0.5 (conservative): ---')
print(generate('once upon a time', n=80, temperature=0.5))
print('\n--- Temperature 1.2 (creative): ---')
print(generate('once upon a time', n=80, temperature=1.2))
print('\n--- Top-k sampling (k=5): ---')
print(generate('roses are red', n=60, temperature=1.0, top_k=5))

## 🌍 Real-World Worked Example — Character-Level Text Generator

**Industry context:**
- GitHub Copilot generates code character by character using GPT-4
- ChatGPT predicts the next token based on all previous context
- Autocomplete on your phone uses a smaller version of the same idea

We build a **character-level language model** that learns to generate text token by token — the exact mechanism behind all LLMs.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training text ────────────────────────────────────────────────────────────
text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep"
)

chars  = sorted(set(text))
c2i    = {c:i for i,c in enumerate(chars)}
i2c    = {i:c for c,i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]

SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc)-SEQ_LEN-1):
    X_list.append(enc[i:i+SEQ_LEN])
    y_list.append(enc[i+SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── LSTM Language Model ───────────────────────────────────────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out,_ = self.lstm(self.embed(x))
        return self.fc(out[:,-1,:])

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]  # mini-batch
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Text Generation (Greedy / Temperature Sampling) ──────────────────────
def generate(seed_str, steps=80, temperature=0.8):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

print("\n── Generated Text ──────────────────────────────────────────────")
print(generate("to be or not", steps=100))
print("\nThis is exactly how ChatGPT generates text — one token at a time.")

## 📚 References & Further Reading

**Papers:**
- Radford et al. (2019) — [GPT-2: Language Models are Unsupervised Multitask Learners](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Brown et al. (2020) — [GPT-3](https://arxiv.org/abs/2005.14165)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)

**State-of-the-Art:** GPT-4, Claude 3.5, Gemini 1.5 — all trained on trillions of tokens with transformer decoders.

## 📝 Summary

In this notebook you studied **08 Generating Creative Text Stories Poems** — a key component of modern AI systems. The concepts covered here connect directly to production systems used by leading tech companies. Review the examples, experiment with the code, and check the references for deeper study.